# 🌿 Hierarchical Clustering
**Module 1 — Clustering Algorithms**

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_blobs, load_iris
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy.spatial.distance import pdist
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded ✅')

## 2. What is Hierarchical Clustering?
> Builds a **tree of clusters (dendrogram)** by either:
- **Agglomerative (bottom-up):** Start with each point as a cluster, merge closest pairs
- **Divisive (top-down):** Start with one cluster, split recursively

**Linkage Criteria (how to measure cluster distance):**
| Linkage | Description |
|---|---|
| **Ward** | Minimizes variance within merged clusters (most common) |
| **Complete** | Distance = max of all pairwise distances |
| **Average** | Distance = avg of all pairwise distances |
| **Single** | Distance = min of all pairwise distances (chaining) |

## 3. Dataset

In [ ]:
X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.9, random_state=42)
print(f'Shape: {X.shape}')

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X[:, 0], X[:, 1], c=y_true, cmap='tab10', s=30, alpha=0.7)
ax.set_title('Raw Data — 4 Clusters', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

## 4. Dendrogram — Ward Linkage

In [ ]:
Z = linkage(X, method='ward')

fig, ax = plt.subplots(figsize=(16, 7))
dendrogram(Z, ax=ax, truncate_mode='lastp', p=30,
           leaf_rotation=90, leaf_font_size=9,
           color_threshold=Z[-4, 2],
           above_threshold_color='gray')
ax.axhline(y=Z[-4, 2], color='red', linestyle='--', lw=1.5, label='Cut at K=4')
ax.set_title('Dendrogram — Ward Linkage (truncated to last 30 merges)', fontsize=14, fontweight='bold')
ax.set_xlabel('Sample Index (or cluster size)', fontsize=11)
ax.set_ylabel('Distance', fontsize=11)
ax.legend(); plt.tight_layout(); plt.show()

## 5. Compare All Linkage Methods

In [ ]:
linkages = ['ward', 'complete', 'average', 'single']
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for i, link in enumerate(linkages):
    Z_link = linkage(X, method=link)
    # Dendrogram
    dendrogram(Z_link, ax=axes[0, i], truncate_mode='lastp', p=20,
               leaf_rotation=90, leaf_font_size=7, no_labels=True)
    axes[0, i].set_title(f'{link.capitalize()} — Dendrogram', fontsize=11, fontweight='bold')

    # Cluster result
    model = AgglomerativeClustering(n_clusters=4, linkage=link)
    labels = model.fit_predict(X)
    sil = silhouette_score(X, labels)
    axes[1, i].scatter(X[:, 0], X[:, 1], c=labels, cmap='tab10', s=20, alpha=0.7)
    axes[1, i].set_title(f'{link.capitalize()}  |  Sil={sil:.3f}', fontsize=11, fontweight='bold')

plt.suptitle('Hierarchical Clustering — Linkage Comparison', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

## 6. Agglomerative Clustering — Best Model (Ward, K=4)

In [ ]:
model = AgglomerativeClustering(n_clusters=4, linkage='ward')
labels = model.fit_predict(X)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, lbl, title in zip(axes, [labels, y_true], ['Hierarchical (Ward, K=4)', 'Ground Truth']):
    ax.scatter(X[:, 0], X[:, 1], c=lbl, cmap='tab10', s=35, alpha=0.7)
    ax.set_title(title, fontsize=13, fontweight='bold')
plt.suptitle('Hierarchical vs Ground Truth', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

## 7. Evaluation Metrics

In [ ]:
results = {}
for link in linkages:
    m = AgglomerativeClustering(n_clusters=4, linkage=link)
    lbl = m.fit_predict(X)
    results[link] = {
        'Silhouette': silhouette_score(X, lbl),
        'Davies-Bouldin': davies_bouldin_score(X, lbl)
    }

results_df = pd.DataFrame(results).T
print('=== Linkage Comparison ===')
print(results_df.round(4))
results_df.plot(kind='bar', figsize=(10, 5), rot=0, colormap='Set2',
                title='Linkage Method Comparison')
plt.tight_layout(); plt.show()

## 8. Choosing K from Dendrogram — Height Differences

In [ ]:
Z = linkage(X, method='ward')
last = Z[-10:, 2]
last_rev = last[::-1]
idxs = np.arange(1, len(last_rev) + 1)
accelerations = np.diff(last_rev, 2)  # 2nd derivative
k_best = accelerations.argmax() + 2

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(idxs, last_rev, 'bo-', markersize=7, lw=2, label='Merge Distance')
ax.axvline(k_best, color='red', linestyle='--', lw=1.5, label=f'Suggested K={k_best}')
ax.set_xlabel('Number of Clusters', fontsize=12)
ax.set_ylabel('Merge Distance', fontsize=12)
ax.set_title('Optimal K via Acceleration of Merge Distances', fontsize=13, fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()
print(f'Suggested optimal K: {k_best}')

## 9. Iris Dataset — Hierarchical Clustering

In [ ]:
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data)

Z_iris = linkage(X_iris, method='ward')
fig, ax = plt.subplots(figsize=(14, 6))
dendrogram(Z_iris, ax=ax, leaf_rotation=90, leaf_font_size=7, color_threshold=5.5)
ax.axhline(5.5, color='red', linestyle='--', lw=1.5, label='Cut → K=3')
ax.set_title('Iris Dataset Dendrogram (Ward Linkage)', fontsize=14, fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

model_iris = AgglomerativeClustering(n_clusters=3, linkage='ward')
lbl_iris = model_iris.fit_predict(X_iris)
X_pca = PCA(n_components=2).fit_transform(X_iris)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, lbl, title in zip(axes, [lbl_iris, iris.target], ['Hierarchical', 'Ground Truth']):
    ax.scatter(X_pca[:, 0], X_pca[:, 1], c=lbl, cmap='Set1', s=40, alpha=0.7)
    ax.set_title(title, fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()
print(f'Silhouette Score (Iris): {silhouette_score(X_iris, lbl_iris):.4f}')

## 10. Key Takeaways
> - Hierarchical clustering gives a **full tree** — no need to pre-specify K
> - **Ward linkage** usually gives the most compact, balanced clusters
> - **Dendrogram** helps visually identify the right number of clusters
> - **Computationally expensive** O(n³) — not ideal for large datasets (>10,000 points)